In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import glob
import seaborn as sns
import sys
import copy
from tqdm.notebook import tqdm
from numba import jit
from scipy import stats
import networkx as nx
import random
import re
from numba import njit

#plt.style.use('seaborn-deep')
plt.rcParams["text.usetex"] = True
plt.rcParams['text.latex.preamble'] = r'\usepackage{amssymb,amsmath}'

plt.rcParams["figure.figsize"] = 11.7, 8.3
plt.rcParams["figure.dpi"] = 75

plt.rcParams["font.size"] = 24
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Fira Sans", 'PT Sans', 'Open Sans', 'Roboto', 'DejaVu Sans', 'Liberation Sans', 'sans-serif']

plt.rcParams["legend.frameon"] = True
plt.rcParams["legend.fancybox"] = True
plt.rcParams["legend.fontsize"] = "small"

plt.rcParams["lines.linewidth"] = 2.5
plt.rcParams["lines.markersize"] = 14
plt.rcParams["lines.markeredgewidth"] = 2

plt.rcParams["xtick.major.size"] = 8
plt.rcParams["ytick.major.size"] = 8

# Taken from https://github.com/giotto-ai/giotto-tda/blob/master/examples/classifying_shapes.ipynb

In [2]:
from generate_datasets import make_point_clouds
num_points = 20
num_samples = 10
point_clouds_basic, labels_basic = make_point_clouds(n_samples_per_shape=num_samples, n_points=num_points, noise=0.5)
point_clouds_basic.shape, labels_basic.shape

((30, 400, 3), (30,))

In [3]:
from gtda.plotting import plot_point_cloud

plot_point_cloud(point_clouds_basic[0])

In [4]:
plot_point_cloud(point_clouds_basic[10])

In [5]:
plot_point_cloud(point_clouds_basic[-1])

# RJ some comments:
We need to have the same shape of `all_diagrams` as in the original code so we can use it in the next pipeline.

```
diagrams_basic.shape = (n_examples, X, n_k_values)

where k_values = [0, 1, 2] 
```

X is a some values extracted from PH (but the size does not matter, I think).

Once we will have `all_diagrams` in a right format we could use it in the function:

```
from gtda.diagrams import PersistenceEntropy

persistence_entropy = PersistenceEntropy()

# calculate topological feature matrix
X_basic = persistence_entropy.fit_transform(diagrams_basic)

# expect shape - (n_point_clouds, n_homology_dims)
X_basic.shape

```

And later we can follow what they did.

DATASET: https://tabula-sapiens.sf.czbiohub.org/organs



In [6]:
from pynndescent import NNDescent
import math

def euclidean_distance_3d(p, q):
     return math.sqrt((p[0] - q[0])**2 + (p[1] - q[1])**2 + (p[2] - q[2])**2)

# function to convert point cloud into graph
def convert_pointcloud_to_graph(point_cloud, graph_type, r=None, k=None):
    if graph_type == 'knn':
        if k is None:
            raise ValueError("Parameter 'k' must be provided for 'knn' graph type.")
        # graph construction using knn

        # Compute kNN using NNDescent
        # indices: array where each row contains the indices of the k-nearest neighbors for each point.
        # I put k+1 because the algorithm counts the same node as the nearest
        knnData = NNDescent(point_cloud, metric="euclidean", n_neighbors=k+1, verbose=True)
        indices, distances = knnData.neighbor_graph

        # Create graph
        G = nx.Graph()

        for i in range(len(point_clouds_basic[0])):
            G.add_node(i, pos=tuple(point_cloud[i]))

        # Add edges based on kNN
        for i in range(len(point_cloud)):
            for j in indices[i]:  
                if i!=j: # this is to exclude self loops
                    G.add_edge(i, j)
        return(G)
        
    elif graph_type == 'rgg':
        if r is None:
            raise ValueError("Parameter 'r' must be provided for 'rgg' graph type.")
        # graph construction using rgg

        G = nx.Graph()

        for i in range(len(point_clouds_basic[0])):
            G.add_node(i, pos=tuple(point_cloud[i]))

        # Add edges based on random geometric graph
        for i in range(len(point_cloud)):
            for j in range(len(point_cloud)):  
                if i!=j:
                    if euclidean_distance_3d(point_cloud[i],point_cloud[j]) < r:
                        G.add_edge(i, j)
        return(G)
    
    else:
        raise ValueError("graph_type must be 'rgg' or 'knn'.")


In [7]:
# The function that takes a graph G and computes persistence diagrams with a given edge or node filtration is already created
# it's this one: calculate_persistence_diagram(G, filtration, k_homology)

from filtrations import density_sum_cycles, ollivier_ricci_curvature, edge_betweenness_centrality, vietoris_rips, laplacian_eigenvalues, degrees
from diagrams import calculate_persistence_diagram

filename = "graphs/point_cloud0_knn_4_degree_khomo_1.edgelist"
G = nx.read_edgelist(filename, nodetype=int)  
persistence_diagram_OR = calculate_persistence_diagram(G, ollivier_ricci_curvature, 1)
persistence_diagram_degrees = calculate_persistence_diagram(G, degrees, 1, True) # Use True for node-based filtrations
# density_sum_cycles requires fortran
# persistence_diagram = calculate_persistence_diagram(G, density_sum_cycles, 1)
persistence_diagram_BC = calculate_persistence_diagram(G, edge_betweenness_centrality, 1)
persistence_diagram_laplacian = calculate_persistence_diagram(G, laplacian_eigenvalues, 1, True)
# I had to modify the function generate_shortest_path_distance_matrix in distances.py to ensure node labels are sequential from 0 to len(G)-1
persistence_diagram_VR = calculate_persistence_diagram(G, vietoris_rips, 1, True)


In [13]:
# function that processes all point clouds
import csv

def get_filtration_name(filtration_type):
    if filtration_type == degrees:
        return 'degree'
    if filtration_type == ollivier_ricci_curvature:
        return 'ORC'
    if filtration_type == edge_betweenness_centrality:
        return 'EBC'
    if filtration_type == vietoris_rips:
        return 'VR'
    if filtration_type == laplacian_eigenvalues:
        return 'laplacian'

"""
# k_values = [4, 5, 6, 7, 8, 9, 10, 20, 30]
# r_values = [0.1, 0.3, 0.5, 0.7, 0.9, 1.1, 1.3, 1.5]
def get_persistent_diagrams_from_graphs(point_cloud, num_cloud, graph_type, filtration_type, k_homology, k_values = [4, 5, 6, 7], r_values = [0.1, 0.3]):
    diagrams = []
    filtration_name = get_filtration_name(filtration_type)
    if graph_type == 'knn':
        for k in k_values:
            G = convert_pointcloud_to_graph(point_cloud, 'knn', None, k)
            
            # Save the graph to an edgelist file
            output_filename = f"graphs/point_cloud{num_cloud}_knn_{k}_{filtration_name}_khomo_{k_homology}.edgelist"
            nx.write_edgelist(G, output_filename, data=False) 
            print(f"Graph saved to {output_filename}")
            if filtration_type in [degrees, vietoris_rips, laplacian_eigenvalues]:
                persistence_diagram = calculate_persistence_diagram(G, filtration_type, k_homology, True)
            else:
                persistence_diagram = calculate_persistence_diagram(G, filtration_type, k_homology)
            diagrams.append(persistence_diagram)

            output_filename_diagram = f"diagrams/point_cloud{num_cloud}_knn_{k}_{filtration_name}_khomo_{k_homology}.csv"
            with open(output_filename_diagram, "w", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(["birth", "death", "k dimension"]) 
                writer.writerows(persistence_diagram)  
            print(f"Diagram saved to {output_filename_diagram}")
        return diagrams
    
    elif graph_type == 'rgg':
        for r in r_values:
            G = convert_pointcloud_to_graph(point_cloud, 'rgg', r, None)
            
            output_filename = f"graphs/point_cloud{num_cloud}_rgg_{r}_{filtration_name}_khomo_{k_homology}.edgelist"
            nx.write_edgelist(G, output_filename, data=False) 
            print(f"Graph saved to {output_filename}")

            if filtration_type in [degrees, vietoris_rips, laplacian_eigenvalues]:
                persistence_diagram = calculate_persistence_diagram(G, filtration_type, k_homology, True)
            else:
                persistence_diagram = calculate_persistence_diagram(G, filtration_type, k_homology)
            diagrams.append(persistence_diagram)

            output_filename_diagram = f"diagrams/point_cloud{num_cloud}_rgg_{r}_{filtration_name}_khomo_{k_homology}.csv"
            with open(output_filename_diagram, "w", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(["birth", "death", "k dimension"]) 
                writer.writerows(persistence_diagram)  
            print(f"Diagram saved to {output_filename_diagram}")
        return diagrams

    else:
        raise ValueError("graph_type must be 'rgg' or 'knn'.")



def get_all_diagrams(point_clouds_basic, filtration_type, graph_type, k_homology_list):
    all_diagrams = []  # List to store diagrams for all point clouds

    for num_cloud in range(len(point_clouds_basic)):
        point_cloud = point_clouds_basic[num_cloud]  
        cloud_diagrams = []  # Collect diagrams for this point cloud
        
        for k_homology in k_homology_list:
            k_diagrams = []  # List to store diagrams for different values of k
            diagrams = get_persistent_diagrams_from_graphs(point_cloud, num_cloud, graph_type, filtration_type, k_homology)
            
            if diagrams:
                for diagram in diagrams:
                    k_diagrams.append(np.array(diagram, dtype=np.float64))  
                cloud_diagrams.append(k_diagrams)  

        all_diagrams.append(cloud_diagrams)
        
    all_diagrams = np.array(all_diagrams, dtype=object)
    all_diagrams.reshape(num_samples, -1, len(k_homology_list))

    return all_diagrams


k_homology_list = [0, 1, 2] 
filtration_type = degrees
graph_type = 'knn'
all_diagrams = get_all_diagrams(point_clouds_basic, filtration_type, graph_type, k_homology_list)
"""

"""
# Add density_sum_cycles
filtrations_list = [degrees, ollivier_ricci_curvature, edge_betweenness_centrality, vietoris_rips, laplacian_eigenvalues]
graph_type_list = ['knn', 'rgg']
# k_homology_list = [0, 1, 2, 3, 4]
# for filtration_type in filtrations_list:
    # for graph_type in graph_type_list: 
"""

"\n# Add density_sum_cycles\nfiltrations_list = [degrees, ollivier_ricci_curvature, edge_betweenness_centrality, vietoris_rips, laplacian_eigenvalues]\ngraph_type_list = ['knn', 'rgg']\n# k_homology_list = [0, 1, 2, 3, 4]\n# for filtration_type in filtrations_list:\n    # for graph_type in graph_type_list: \n"

In [44]:
def compute_diagrams_from_point_clouds(point_clouds, filtration_type, graph_type, graph_parameter, k_homology):
    all_diagrams = []
    for p in tqdm(point_clouds):
        g = convert_pointcloud_to_graph(p, graph_type, k=graph_parameter)
        if filtration_type in [degrees, vietoris_rips, laplacian_eigenvalues]:
            d = calculate_persistence_diagram(g, filtration_type, k_homology, True)
        else:
            d = calculate_persistence_diagram(g, filtration_type, k_homology)
        all_diagrams.append(d)
#     return np.array(all_diagrams)
    return all_diagrams

tmp_diagrams = compute_diagrams_from_point_clouds(point_clouds_basic, degrees, 'knn', 5, 2)

  0%|          | 0/30 [00:00<?, ?it/s]

Fri Feb 14 17:19:02 2025 Building RP forest with 9 trees
Fri Feb 14 17:19:02 2025 NN descent for 9 iterations
	 1  /  9
	 2  /  9
	Stopping threshold met -- exiting after 2 iterations
Fri Feb 14 17:19:02 2025 Building RP forest with 9 trees
Fri Feb 14 17:19:02 2025 NN descent for 9 iterations
	 1  /  9
	 2  /  9
	Stopping threshold met -- exiting after 2 iterations
Fri Feb 14 17:19:02 2025 Building RP forest with 9 trees
Fri Feb 14 17:19:02 2025 NN descent for 9 iterations
	 1  /  9
	 2  /  9
	Stopping threshold met -- exiting after 2 iterations
Fri Feb 14 17:19:02 2025 Building RP forest with 9 trees
Fri Feb 14 17:19:02 2025 NN descent for 9 iterations
	 1  /  9
	 2  /  9
	Stopping threshold met -- exiting after 2 iterations
Fri Feb 14 17:19:02 2025 Building RP forest with 9 trees
Fri Feb 14 17:19:02 2025 NN descent for 9 iterations
	 1  /  9
	 2  /  9
	Stopping threshold met -- exiting after 2 iterations
Fri Feb 14 17:19:02 2025 Building RP forest with 9 trees
Fri Feb 14 17:19:02 202

In [45]:
[t.shape for t in tmp_diagrams]

[(583, 3),
 (574, 3),
 (503, 3),
 (514, 3),
 (521, 3),
 (562, 3),
 (528, 3),
 (502, 3),
 (514, 3),
 (534, 3),
 (439, 3),
 (473, 3),
 (453, 3),
 (482, 3),
 (474, 3),
 (442, 3),
 (474, 3),
 (485, 3),
 (473, 3),
 (480, 3),
 (379, 3),
 (425, 3),
 (409, 3),
 (378, 3),
 (393, 3),
 (375, 3),
 (389, 3),
 (388, 3),
 (386, 3),
 (406, 3)]

In [16]:
from gtda.homology import VietorisRipsPersistence

# Track connected components, loops, and voids
homology_dimensions = [0, 1, 2]

# Collapse edges to speed up H2 persistence calculation!
persistence = VietorisRipsPersistence(
    metric="euclidean",
    homology_dimensions=homology_dimensions,
    n_jobs=6,
    collapse_edges=True,
)

diagrams_basic = persistence.fit_transform(point_clouds_basic)
print(diagrams_basic)

[[[0.00000000e+00 9.39698890e-04 0.00000000e+00]
  [0.00000000e+00 2.41730874e-03 0.00000000e+00]
  [0.00000000e+00 4.13253810e-03 0.00000000e+00]
  ...
  [1.12038024e-01 1.12038024e-01 2.00000000e+00]
  [1.12038024e-01 1.12038024e-01 2.00000000e+00]
  [1.12038024e-01 1.12038024e-01 2.00000000e+00]]

 [[0.00000000e+00 2.22813641e-03 0.00000000e+00]
  [0.00000000e+00 4.70298156e-03 0.00000000e+00]
  [0.00000000e+00 4.82005952e-03 0.00000000e+00]
  ...
  [1.12038024e-01 1.12038024e-01 2.00000000e+00]
  [1.12038024e-01 1.12038024e-01 2.00000000e+00]
  [1.12038024e-01 1.12038024e-01 2.00000000e+00]]

 [[0.00000000e+00 2.50579836e-03 0.00000000e+00]
  [0.00000000e+00 3.62749747e-03 0.00000000e+00]
  [0.00000000e+00 5.05612930e-03 0.00000000e+00]
  ...
  [1.12038024e-01 1.12038024e-01 2.00000000e+00]
  [1.12038024e-01 1.12038024e-01 2.00000000e+00]
  [1.12038024e-01 1.12038024e-01 2.00000000e+00]]

 ...

 [[0.00000000e+00 5.89905344e-02 0.00000000e+00]
  [0.00000000e+00 6.01881444e-02 0.0000

In [46]:
print(tmp_diagrams)

[array([[ 5., inf,  0.],
       [ 5.,  8.,  0.],
       [ 5.,  8.,  0.],
       ...,
       [ 9., inf,  2.],
       [ 9., inf,  2.],
       [ 9., inf,  2.]]), array([[ 5., inf,  0.],
       [ 5.,  7.,  0.],
       [ 5.,  7.,  0.],
       ...,
       [ 8., inf,  2.],
       [ 8., inf,  2.],
       [ 8., inf,  2.]]), array([[ 5., inf,  0.],
       [ 5.,  7.,  0.],
       [ 5.,  7.,  0.],
       ...,
       [ 8., inf,  2.],
       [ 8., inf,  2.],
       [ 8., inf,  2.]]), array([[ 5., inf,  0.],
       [ 5.,  7.,  0.],
       [ 5.,  7.,  0.],
       ...,
       [ 9., inf,  2.],
       [ 9., inf,  2.],
       [ 9., inf,  2.]]), array([[ 5., inf,  0.],
       [ 5.,  7.,  0.],
       [ 5.,  7.,  0.],
       ...,
       [ 8., inf,  2.],
       [ 8., inf,  2.],
       [ 8., inf,  2.]]), array([[ 5., inf,  0.],
       [ 5., 11.,  0.],
       [ 5.,  7.,  0.],
       ...,
       [ 9., inf,  2.],
       [ 9., inf,  2.],
       [ 9., inf,  2.]]), array([[ 5., inf,  0.],
       [ 5.,  7.,  0.],
   

In [47]:
def totalpersistence_entropy(diagram, k):
    filtered_points = diagram[diagram[:, 2] == k] #Gets only the points of the diagram of dimension k

    lifetimes = filtered_points[:, 1] - filtered_points[:, 0]
    lifetimes = lifetimes[np.isfinite(lifetimes)]  # Remove points with infinite death

    # If no valid lifetimes, return entropy as 0
    if len(lifetimes) == 0:
        return 0, 0

    total_persistence = np.sum(lifetimes)
    probabilities = lifetimes / total_persistence

    entropy = -np.sum(probabilities * np.log(probabilities))
    return total_persistence, entropy

# Compute entropy for each diagram and each dimension
total_persistence_values = []
entropy_values = []
for diagram in tmp_diagrams:
    TP_k0, entropy_k0 = totalpersistence_entropy(diagram, 0)
    TP_k1, entropy_k1 = totalpersistence_entropy(diagram, 1)
    TP_k2, entropy_k2 = totalpersistence_entropy(diagram, 2)
    total_persistence_values.append((TP_k0, TP_k1, TP_k2))
    entropy_values.append((entropy_k0, entropy_k1, entropy_k2))

entropy_values = np.array(entropy_values)
total_persistence_values = np.array(total_persistence_values)
print(total_persistence_values)
print(entropy_values)

[[96. 17.  0.]
 [68. 18.  0.]
 [75.  7.  0.]
 [71.  9.  0.]
 [69. 18.  0.]
 [79. 22.  0.]
 [74. 12.  0.]
 [73. 12.  0.]
 [58. 14.  0.]
 [69. 17.  0.]
 [75. 26.  0.]
 [71. 30.  0.]
 [61. 29.  0.]
 [66. 27.  0.]
 [77. 19.  0.]
 [80. 29.  0.]
 [67. 19.  0.]
 [58. 14.  0.]
 [76. 26.  0.]
 [78. 17.  0.]
 [47.  9.  0.]
 [58.  9.  0.]
 [50.  9.  0.]
 [54. 10.  0.]
 [54. 14.  0.]
 [57.  8.  0.]
 [55. 15.  0.]
 [57. 13.  0.]
 [61.  4.  0.]
 [57.  7.  0.]]
[[4.17254196 2.23160695 0.        ]
 [3.99525421 2.50528999 0.        ]
 [4.16961672 1.94591015 0.        ]
 [3.96980079 1.88915916 0.        ]
 [4.17383284 2.47622065 0.        ]
 [4.0227885  2.32654325 0.        ]
 [4.09799431 2.13833306 0.        ]
 [4.08156577 2.02280853 0.        ]
 [3.82142674 2.30461938 0.        ]
 [4.0532855  2.0685135  0.        ]
 [4.11416494 2.57801851 0.        ]
 [4.06742715 2.40723472 0.        ]
 [3.92906477 2.87122025 0.        ]
 [4.06362798 2.56765936 0.        ]
 [4.18177102 2.3786202  0.        ]
 [4.17408

In [48]:
point_clouds_basic.shape

(30, 400, 3)

In [49]:
diagrams_basic.shape

(30, 593, 3)

In [31]:
from gtda.diagrams import PersistenceEntropy

persistence_entropy = PersistenceEntropy()

# calculate topological feature matrix
X_basic = persistence_entropy.fit_transform(diagrams_basic)

# expect shape - (n_point_clouds, n_homology_dims)
X_basic.shape

(30, 3)

In [33]:
print(X_basic)
plot_point_cloud(X_basic)

[[8.46036712 4.21253335 1.95162516]
 [8.4666803  4.4248263  1.17921022]
 [8.47145994 4.48364    1.27635212]
 [8.43942946 4.32503151 1.79187947]
 [8.46947157 4.51852348 1.27320909]
 [8.44461994 4.26716124 1.97128971]
 [8.48223828 4.07756127 1.18064099]
 [8.46746129 4.6992853  1.54999523]
 [8.44138119 4.40870649 1.81931596]
 [8.457638   4.39604536 0.91076054]
 [8.53785315 6.4274824  1.1950708 ]
 [8.53529943 6.5707866  1.61616478]
 [8.53870579 6.70057565 1.43764369]
 [8.52832796 6.75594436 1.91543695]
 [8.51801867 6.77546335 1.8711963 ]
 [8.53568073 6.79276453 1.87614318]
 [8.51890794 6.58993987 1.84430635]
 [8.51145652 6.54082847 1.37834834]
 [8.52008688 6.54580421 1.7360317 ]
 [8.51476075 6.50366021 1.99257007]
 [8.54577709 6.29354127 3.4938417 ]
 [8.54342715 6.48570874 3.66336556]
 [8.54097075 6.4170831  3.40882843]
 [8.56006373 6.35992129 3.59125254]
 [8.55827243 6.41953318 3.53878831]
 [8.55076983 6.3247436  3.46678759]
 [8.54913237 6.33592063 3.4992208 ]
 [8.55499076 6.46180447 3.70

In [38]:
plot_point_cloud(entropy_values)

In [34]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(oob_score=True)
rf.fit(X_basic, labels_basic)

print(f"OOB score: {rf.oob_score_:.3f}")

OOB score: 1.000


In [50]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(oob_score=True)
rf.fit(total_persistence_values, labels_basic)

print(f"OOB score: {rf.oob_score_:.3f}")

OOB score: 0.800


In [51]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(oob_score=True)
rf.fit(entropy_values, labels_basic)

print(f"OOB score: {rf.oob_score_:.3f}")

OOB score: 0.633


In [42]:
from gtda.pipeline import Pipeline

steps = [
    ("persistence", VietorisRipsPersistence(metric="euclidean", homology_dimensions=homology_dimensions, n_jobs=6)),
    ("entropy", PersistenceEntropy()),
    ("model", RandomForestClassifier(oob_score=True)),
]

pipeline = Pipeline(steps)

In [14]:
pipeline.fit(point_clouds_basic, labels_basic)

Pipeline(steps=[('persistence',
                 VietorisRipsPersistence(homology_dimensions=[0, 1, 2],
                                         n_jobs=6)),
                ('entropy', PersistenceEntropy()),
                ('model', RandomForestClassifier(oob_score=True))])

In [15]:
pipeline["model"].oob_score_

1.0

---
---

In [16]:
from openml.datasets.functions import get_dataset

df = get_dataset('shapes').get_data(dataset_format='dataframe')[0]
df.head()

,x,y,z,target
0,0.341007,0.318606,0.096725,human_arms_out9
1,0.329226,0.421601,0.056749,human_arms_out9
2,0.446869,0.648674,0.124090,human_arms_out9
3,0.314729,0.217860,0.070847,human_arms_out9
4,0.426678,0.919195,0.047609,human_arms_out9


In [17]:
import openml

In [18]:
plot_point_cloud(df.query('target == "biplane0"')[["x", "y", "z"]].values)

In [19]:
import numpy as np

point_clouds = np.asarray(
    [
        df.query("target == @shape")[["x", "y", "z"]].values
        for shape in df["target"].unique()
    ]
)
point_clouds.shape

(40, 400, 3)

In [20]:
persistence = VietorisRipsPersistence(
    metric="euclidean",
    homology_dimensions=homology_dimensions,
    n_jobs=6,
    collapse_edges=True,
)
persistence_diagrams = persistence.fit_transform(point_clouds)

In [21]:
# Index - (human_arms_out, 0), (vase, 10), (dining_chair, 20), (biplane, 30)
index = 30
plot_diagram(persistence_diagrams[index])

In [22]:
persistence_entropy = PersistenceEntropy(normalize=True)
# Calculate topological feature matrix
X = persistence_entropy.fit_transform(persistence_diagrams)
# Visualise feature matrix
plot_point_cloud(X)

In [23]:
labels = np.zeros(40)
labels[10:20] = 1
labels[20:30] = 2
labels[30:] = 3

rf = RandomForestClassifier(oob_score=True, random_state=42)
rf.fit(X, labels)
rf.oob_score_

0.6